# VDM for generative modeling

This notebook implements the Variational Diffusion Model (VDM), a special case of the SDE framework from notebook 02 with a fixed (non-neural) variational distribution.

The variational marginal is restricted to
$$
q_t(x|y) = \mathcal{N}\!\left(x \mid \alpha(t)\,y,\, \beta^2(t)\right),
$$
where $\alpha(t)$ and $\beta(t)$ are **fixed** scalar functions of $t$ satisfying variance preservation $\alpha^2(t)+\beta^2(t)=1$. Here **$t=0$ is the prior** ($\alpha(0)=0$, $\beta(0)=1$, so $q_0(x|y)=\mathcal{N}(0,1)$) and **$t=1$ is the data** ($\alpha(1)\approx 1$, $\beta(1)\approx 0$), consistent with notebooks 01–03.

The SNR schedule is linear in log space with learnable endpoints:
$$
\log \mathrm{SNR}(t) = (1-t)\,\log \mathrm{SNR}(0) + t\,\log \mathrm{SNR}(1),
\qquad
\alpha(t) = \sqrt{\frac{\mathrm{SNR}(t)}{1+\mathrm{SNR}(t)}},
\quad
\beta(t) = \frac{1}{\sqrt{1+\mathrm{SNR}(t)}}.
$$
with $\log\mathrm{SNR}(0)\ll 0$ (noise) and $\log\mathrm{SNR}(1)\gg 0$ (data).

The generative drift is parameterised via a **signal prediction network** $\widehat{y}(x_t, t)$:
$$
\overrightarrow{f}(x,t) = \overrightarrow{g}(x,\,\widehat{y}(x,t),\,t),
$$
and the ELBO drift term simplifies to the SNR-weighted objective
$$
\mathcal{D} = \frac{1}{2}\int_0^1 \left|\frac{\partial\,\mathrm{SNR}(t)}{\partial t}\right|
\mathbb{E}_{q_t(x|y)}\!\left[\|y - \widehat{y}(x_t,t)\|^2\right]\mathrm{d}t.
$$

The full ELBO is
$$
\log p(y) \ge
\underbrace{\mathbb{E}_{q_1}[\log p(y|x_1)]}_{\text{likelihood at }t=1}
- \underbrace{\mathrm{KL}(q_0(x|y)\|\,p_0(x))}_{=\,0\text{ exactly (both }=\mathcal{N}(0,1))}
- \mathcal{D}.
$$

Generation runs the **forward generative SDE** from $x_0 \sim \mathcal{N}(0,1)$ at $t=0$ to $x_1 \approx y$ at $t=1$.

## Setup

In [10]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system("wget -q https://raw.githubusercontent.com/olewinther/generative-ode-sde/main/utils.py")
else:
    for path in ['..', '.']:
        if os.path.exists(os.path.join(path, 'utils.py')):
            sys.path.insert(0, os.path.abspath(path))
            break

from utils import *

## GPU

In [11]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU State:', device)

GPU State: cpu


## SNR schedule

The schedule holds $\log\mathrm{SNR}(0)$ and $\log\mathrm{SNR}(1)$ (learnable or fixed) and derives $\alpha(t)$, $\beta(t)$, and the diffusion coefficient from first principles:
$$
\sigma^2(t) = \alpha^2(t)\,\frac{\partial}{\partial t}\!\left[\frac{\beta^2(t)}{\alpha^2(t)}\right]
= \bigl(\log\mathrm{SNR}(0)-\log\mathrm{SNR}(1)\bigr)\,\beta^2(t).
$$
In our convention $\log\mathrm{SNR}(0)\ll 0 < \log\mathrm{SNR}(1)$, so $\sigma^2(t)<0$. The diffusion coefficient for the generative SDE is $\tilde\sigma(t)=\sqrt{-\sigma^2(t)}>0$.

In [12]:
import torch.nn as nn

class SNRSchedule(nn.Module):
    """
    Linear log-SNR schedule with variance preservation.
    log_snr_0: log SNR at t=0 (noise end, low SNR).
    log_snr_1: log SNR at t=1 (data end, high SNR).
    Set trainable=True to learn the endpoints jointly with the model.
    """
    def __init__(self, log_snr_0=-10.0, log_snr_1=10.0, trainable=False):
        super().__init__()
        for name, val in [('log_snr_0', log_snr_0), ('log_snr_1', log_snr_1)]:
            buf = torch.tensor(float(val))
            if trainable:
                setattr(self, name, nn.Parameter(buf))
            else:
                self.register_buffer(name, buf)

    def log_snr(self, t):
        return (1 - t) * self.log_snr_0 + t * self.log_snr_1

    def snr(self, t):
        return torch.exp(self.log_snr(t))

    def alpha(self, t):
        return torch.sqrt(self.snr(t) / (1 + self.snr(t)))

    def beta(self, t):
        return torch.sqrt(1 / (1 + self.snr(t)))

    def sigma_sq(self, t):
        # σ²(t) = α²(t) · d/dt[β²/α²] = (log_snr_0 - log_snr_1) · β²(t)  [< 0 in our convention]
        return (self.log_snr_0 - self.log_snr_1) * self.beta(t) ** 2

    def d_log_alpha_dt(self, t):
        # d/dt log α(t) = (log_snr_1 - log_snr_0) / 2 · β²(t)  [> 0]
        return 0.5 * (self.log_snr_1 - self.log_snr_0) * self.beta(t) ** 2

    def d_alpha_dt(self, t):
        return self.alpha(t) * self.d_log_alpha_dt(t)

    def d_beta_dt(self, t):
        # from VP: α² + β² = 1  →  d_β/dt = −α/β · d_α/dt  [< 0]
        return -self.alpha(t) / self.beta(t) * self.d_alpha_dt(t)

    def d_snr_dt(self, t):
        # d/dt SNR(t) = (log_snr_1 - log_snr_0) · SNR(t)  [> 0]
        return (self.log_snr_1 - self.log_snr_0) * self.snr(t)


class VDMAlpha(nn.Module):
    """Variational mean: alpha(y, t) = schedule.alpha(t) * y"""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, y, t):
        return self.schedule.alpha(t) * y


class VDMBeta(nn.Module):
    """Variational std: beta(y, t) = schedule.beta(t)"""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, y, t):
        return self.schedule.beta(t).expand_as(y)


class VDMSigma(nn.Module):
    """Diffusion coefficient for the generative SDE: σ̃(x,t) = sqrt(-sigma_sq(t))"""
    def __init__(self, schedule):
        super().__init__()
        self.schedule = schedule

    def forward(self, x, t):
        return torch.sqrt(-self.schedule.sigma_sq(t)).expand_as(x)

## Signal prediction network

In [13]:
class SignalPredictionNetwork(nn.Module):
    """Predicts y from noisy observation x_t and time t."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=-1))

In [14]:
class Beta(nn.Module):
    def __init__(self, initial_beta=1e-1, trainable=True):
        super().__init__()
        log_b = torch.log(torch.tensor(initial_beta))
        if trainable:
            self.log_beta = nn.Parameter(log_b)
        else:
            self.register_buffer("log_beta", log_b)

    def forward(self):
        return torch.exp(self.log_beta)


class Likelihood(nn.Module):
    def __init__(self, beta):
        super().__init__()
        self.beta = beta

    def log_prob(self, y, x):
        var = self.beta() ** 2
        return -0.5 * ((y - x) ** 2 / var + torch.log(2 * torch.pi * var))

    def sample(self, x):
        return x + torch.randn_like(x) * self.beta()

## Visualize schedule and learned signal prediction

In [15]:
def plot_schedule(schedule):
    t_grid = torch.linspace(0, 1, 200).unsqueeze(1)
    with torch.no_grad():
        alpha_vals = [schedule.alpha(t).item() for t in t_grid]
        beta_vals  = [schedule.beta(t).item()  for t in t_grid]
        snr_vals   = [schedule.snr(t).item()   for t in t_grid]

    fig, axes = plt.subplots(3, 1, figsize=(10, 9))
    axes[0].plot(t_grid.numpy(), alpha_vals)
    axes[0].set_title(r"$\alpha(t)$"); axes[0].set_xlabel("t"); axes[0].set_ylabel(r"$\alpha(t)$")
    axes[1].plot(t_grid.numpy(), beta_vals)
    axes[1].set_title(r"$\beta(t)$");  axes[1].set_xlabel("t"); axes[1].set_ylabel(r"$\beta(t)$")
    axes[2].semilogy(t_grid.numpy(), snr_vals)
    axes[2].set_title("SNR(t)");        axes[2].set_xlabel("t"); axes[2].set_ylabel("SNR(t)")
    plt.tight_layout(); plt.show()


def plot_y_hat(y_hat_net, schedule, num_samples=3):
    t_grid = torch.linspace(0, 1, 100).unsqueeze(1)
    x_samples = torch.randn(num_samples, 1)

    fig, ax = plt.subplots(figsize=(10, 3))
    for s in x_samples:
        with torch.no_grad():
            vals = [y_hat_net(s, t.expand_as(s)).item() for t in t_grid]
        ax.plot(t_grid.numpy(), vals, label=f"x={s.item():.2f}")
    ax.set_title(r"$\widehat{y}(x,t)$"); ax.set_xlabel("t"); ax.set_ylabel(r"$\widehat{y}$"); ax.legend()
    plt.tight_layout(); plt.show()

## VDM ELBO and training loop

The ELBO boundary terms match the convention in notebook 02: the **likelihood is evaluated at $t=1$** (data end) and the **KL is at $t=0$** (noise end). Since $q_0(x|y)=\mathcal{N}(0,1)=p_0(x)$ exactly, the KL vanishes identically — it is retained here for reporting purposes only.

In [16]:
def compute_elbo_vdm(y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood, y, t_sample):
    # 1. Likelihood term at t=1: E_{q_1(x|y)}[log p(y|x_1)]
    q_x1 = Variational(vdm_alpha, vdm_beta, y, torch.ones_like(t_sample))
    x_1 = q_x1.sample()
    likelihood_term = likelihood.log_prob(y, x_1)

    # 2. KL at t=0: KL(q_0(x|y) || p_0(x))  [= 0 exactly since both are N(0,1)]
    q_x0 = Variational(vdm_alpha, vdm_beta, y, torch.zeros_like(t_sample))
    x_0 = q_x0.sample()
    kl_divergence = q_x0.log_prob(x_0) - prior.log_prob(x_0)

    # 3. SNR-weighted signal prediction loss
    eps = torch.randn_like(y)
    x_t = schedule.alpha(t_sample) * y + schedule.beta(t_sample) * eps
    y_hat = y_hat_net(x_t, t_sample)
    drift_term = 0.5 * torch.abs(schedule.d_snr_dt(t_sample)) * (y - y_hat) ** 2

    elbo = likelihood_term - kl_divergence - drift_term
    return elbo.mean(), likelihood_term.mean(), kl_divergence.mean(), drift_term.mean()


def training_loop_vdm(y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood,
                      data_loader, validation_data, n_epochs=1000, lr=1e-3):
    from collections import deque
    trainable = (list(y_hat_net.parameters()) + list(schedule.parameters()) +
                 list(likelihood.beta.parameters()))
    optimizer = torch.optim.Adam(trainable, lr=lr)
    train_history, val_history = deque(maxlen=5), deque(maxlen=5)

    for epoch in range(n_epochs):
        total_loss = 0.0
        for y_batch in data_loader:
            optimizer.zero_grad()
            t_sample = torch.rand(y_batch.size())
            eps = torch.randn_like(y_batch)
            x_t = schedule.alpha(t_sample) * y_batch + schedule.beta(t_sample) * eps
            y_hat = y_hat_net(x_t, t_sample)
            loss = (0.5 * torch.abs(schedule.d_snr_dt(t_sample)) * (y_batch - y_hat) ** 2).mean()
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        if epoch % 50 == 0 or epoch == n_epochs - 1:
            t_val = torch.rand(validation_data.size())
            elbo_val, ll, kl, dm = compute_elbo_vdm(
                y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood,
                validation_data, t_val)
            train_cur = total_loss / len(data_loader)
            val_cur = elbo_val.item()
            train_history.append(train_cur)
            val_history.append(val_cur)
            avg_train = sum(train_history) / len(train_history)
            avg_val   = sum(val_history)   / len(val_history)
            print(f"Epoch {epoch:4d} | train {train_cur:.4f} (avg {avg_train:.4f}) | "
                  f"val {val_cur:.4f} (avg {avg_val:.4f}) | "
                  f"ll={ll:.4f}, kl={kl:.4f}, dm={dm:.4f} | "
                  f"log_snr_0={schedule.log_snr_0.item():.2f} "
                  f"log_snr_1={schedule.log_snr_1.item():.2f}")

    return y_hat_net, schedule

## Generation (forward generative SDE) and visualization

Generation runs the **forward generative SDE** from $x_0\sim\mathcal{N}(0,1)$ at $t=0$ to $x_1\approx y$ at $t=1$.
The noising SDE (data→noise, running backward in our time) has drift $\frac{\partial\log\alpha}{\partial t}\cdot x$ and diffusion $\sigma^2(t)<0$. Anderson's reverse-time formula gives the generative drift:
$$
\overrightarrow{g}(x,\hat y,t)
= \frac{\partial\log\alpha}{\partial t}\cdot x
+ \sigma^2(t)\cdot\frac{x - \alpha(t)\hat y}{\beta^2(t)},
$$
where $\sigma^2(t)<0$ so the second term attracts $x$ toward $\alpha(t)\hat y$. The diffusion coefficient is $\tilde\sigma(t)=\sqrt{-\sigma^2(t)}>0$.

The backward ODE drift (for encoding data to the prior) is
$$
\overleftarrow{g}(x,\hat y,t) = \dot\alpha(t)\,\hat y + \dot\beta(t)\,\frac{x - \alpha(t)\hat y}{\beta(t)}.
$$
Both are computed analytically from the SNR schedule — no autodiff required.

In [17]:
def make_vdm_forward_drift(y_hat_net, schedule):
    """
    Generative drift from Anderson's reverse-SDE applied to the noising process.
    f_noising = d_log_alpha/dt * x; sigma_sq < 0 in our convention.
    Result: d_log_alpha * x + sigma_sq * (x - alpha*yhat) / beta²
    Since sigma_sq < 0, the correction attracts x toward alpha*yhat.
    """
    def drift(x, t):
        y_hat = y_hat_net(x, t)
        alpha = schedule.alpha(t)
        beta = schedule.beta(t)
        d_log_alpha = schedule.d_log_alpha_dt(t)
        sigma_sq = schedule.sigma_sq(t)          # negative in our convention
        return d_log_alpha * x + sigma_sq * (x - alpha * y_hat) / beta ** 2
    return drift


def make_vdm_backward_drift(y_hat_net, schedule):
    """Backward ODE drift for encoding: d_alpha/dt * yhat + d_beta/dt * (x - alpha*yhat) / beta"""
    def drift(x, t):
        y_hat = y_hat_net(x, t)
        alpha = schedule.alpha(t)
        beta = schedule.beta(t)
        d_alpha = schedule.d_alpha_dt(t)
        d_beta = schedule.d_beta_dt(t)
        return d_alpha * y_hat + d_beta * (x - alpha * y_hat) / beta
    return drift

## Create training and validation data

In [18]:
torch.manual_seed(42)
np.random.seed(42)

n_samples, n_val = 1000, 8000

# Choose distribution: 'gaussian', 'laplace', 'laplace_mixture'
training_set_dist = 'laplace_mixture'

if training_set_dist == 'gaussian':
    params = {'mean': torch.tensor(-1.0), 'std': torch.tensor(2.0)}
elif training_set_dist == 'laplace':
    params = {'loc': torch.tensor(0.0), 'scale': torch.tensor(1.0)}
elif training_set_dist == 'laplace_mixture':
    params = {'k': 5, 'spacing': 4.0, 'scale': torch.tensor(1.0)}

training_set = TrainingSetWithLogLikelihood(training_set_dist, params)
training_data, ell_train = training_set.generate_training_data(n_samples)
validation_data, ell_val = training_set.generate_training_data(n_val)
print(f"{training_set_dist} | true log-likelihood: train={ell_train:.4f}, val={ell_val:.4f}")

laplace_mixture | true log-likelihood: train=-2.9829, val=-2.9862


## Run training

In [ ]:
data_loader = torch.utils.data.DataLoader(training_data, batch_size=125, shuffle=True)

schedule = SNRSchedule(log_snr_0=-10.0, log_snr_1=10.0, trainable=False)

y_hat_net  = SignalPredictionNetwork()
vdm_alpha  = VDMAlpha(schedule)
vdm_beta   = VDMBeta(schedule)
vdm_sigma  = VDMSigma(schedule)

prior = Prior(gaussian_sample, gaussian_log_pdf)

delta = Beta(initial_beta=1e-1, trainable=True)
likelihood = Likelihood(beta=delta)

t = torch.linspace(0, 1, steps=100)

vdm_gen_drift = make_vdm_forward_drift(y_hat_net, schedule)
vdm_bwd_drift = make_vdm_backward_drift(y_hat_net, schedule)

forward_path  = ForwardPath(mode="sde", f_net=vdm_gen_drift, sigma_net=vdm_sigma, prior=prior)
backward_path = BackwardPath(mode="ode", f_net=vdm_bwd_drift)

print("Before training:")
plot_schedule(schedule)
plot_y_hat(y_hat_net, schedule)
visualize_paths_and_marginals(validation_data, t, backward_path, forward_path)

y_hat_net, schedule = training_loop_vdm(
    y_hat_net, vdm_alpha, vdm_beta, schedule, prior, likelihood,
    data_loader, validation_data, n_epochs=1000, lr=1e-3,
)

print("After training:")
plot_schedule(schedule)
plot_y_hat(y_hat_net, schedule)
visualize_paths_and_marginals(validation_data, t, backward_path, forward_path)